In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

dataset_path = "/content/drive/MyDrive/Glaucoma_Project"

print(os.listdir(dataset_path))

['split', 'mobilenetv2_model.keras', 'app.py']


In [4]:
import os

split_path = "/content/drive/MyDrive/Glaucoma_Project/split"

print(os.listdir(split_path))

['train', 'test', 'val']


In [5]:
import os

dataset_path = "/content/drive/MyDrive/Glaucoma_Project/split"

for folder in ["train", "test", "val"]:
    print(f"\n{folder.upper()}")

    folder_path = os.path.join(dataset_path, folder)

    for class_name in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_name)

        if os.path.isdir(class_path):
            print(class_name, ":", len(os.listdir(class_path)), "images")


TRAIN
cataract : 727 images
normal : 752 images
glaucoma : 705 images
diabetic_retinopathy : 765 images

TEST
normal : 108 images
cataract : 105 images
glaucoma : 102 images
diabetic_retinopathy : 111 images

VAL
diabetic_retinopathy : 218 images
glaucoma : 201 images
normal : 215 images
cataract : 207 images


In [6]:
import os

dataset_path = "/content/drive/MyDrive/Glaucoma_Project/split"

for split in ["train", "test", "val"]:
    print(f"\n{split.upper()}")

    for class_name in ["glaucoma", "normal"]:
        folder = os.path.join(dataset_path, split, class_name)

        print(
            class_name,
            "exists:", os.path.exists(folder),
            "images:", len(os.listdir(folder)) if os.path.exists(folder) else 0
        )


TRAIN
glaucoma exists: True images: 705
normal exists: True images: 752

TEST
glaucoma exists: True images: 102
normal exists: True images: 108

VAL
glaucoma exists: True images: 201
normal exists: True images: 215


In [7]:
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Dataset path
dataset_path = "/content/drive/MyDrive/Glaucoma_Project/split"

# Image settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training data
train_datagen = ImageDataGenerator(
    rescale=1./255
)

train_data = train_datagen.flow_from_directory(
    dataset_path + "/train",
    classes=["glaucoma", "normal"],
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)

# Validation data
val_datagen = ImageDataGenerator(
    rescale=1./255
)

val_data = val_datagen.flow_from_directory(
    dataset_path + "/val",
    classes=["glaucoma", "normal"],
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

# Test data
test_datagen = ImageDataGenerator(
    rescale=1./255
)

test_data = test_datagen.flow_from_directory(
    dataset_path + "/test",
    classes=["glaucoma", "normal"],
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("\nClass labels:", train_data.class_indices)

Found 1455 images belonging to 2 classes.
Found 416 images belonging to 2 classes.
Found 210 images belonging to 2 classes.

Class labels: {'glaucoma': 0, 'normal': 1}


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    GlobalAveragePooling2D,
    Dropout,
    Dense
)

# Load MobileNetV2 pretrained model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False

# Build our Glaucoma Detection model
model = Sequential([
    Input(shape=(224, 224, 3)),

    # Convert pixel values from [0,1] to [-1,1]
    Rescaling(scale=2.0, offset=-1.0),

    base_model,

    GlobalAveragePooling2D(),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display model architecture
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
# Train the MobileNetV2 model

EPOCHS = 10

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 847s 18s/step - accuracy: 0.7189 - loss: 0.5474 - val_accuracy: 0.8438 - val_loss: 0.3909
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 357ms/step - accuracy: 0.8227 - loss: 0.3973 - val_accuracy: 0.8486 - val_loss: 0.3394
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 363ms/step - accuracy: 0.8426 - loss: 0.3592 - val_accuracy: 0.8582 - val_loss: 0.3296
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 370ms/step - accuracy: 0.8515 - loss: 0.3314 - val_accuracy: 0.8438 - val_loss: 0.3276
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 372ms/step - accuracy: 0.8680 - loss: 0.3265 - val_accuracy: 0.8582 - val_loss: 0.3080
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 16s 356ms/step - accuracy: 0.8687 - loss: 0.3059 - val_accuracy: 0.8726 - val_loss: 0.3046
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 16s 354ms/step - accuracy: 0.8749 - loss: 0.3065 - val_accuracy: 0.8678 - val_loss: 0.3017
Epoch 8/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 363ms/step - accuracy: 0.8701 - loss: 0.2967 - val_accur

In [ ]:
# Evaluate the trained model using test images

test_loss, test_accuracy = model.evaluate(test_data)

print("Test Accuracy:", round(test_accuracy * 100, 2), "%")
print("Test Loss:", round(test_loss, 4))

5/7 ━━━━━━━━━━━━━━━━━━━━ 22s 11s/step - accuracy: 0.8976 - loss: 0.2775

In [1]:
# Evaluate the trained model using test images

test_loss, test_accuracy = model.evaluate(test_data)

print("Test Accuracy:", round(test_accuracy * 100, 2), "%")
print("Test Loss:", round(test_loss, 4))

NameError: name 'model' is not defined

In [4]:
print("Model exists:", "model" in globals())
print("Test data exists:", "test_data" in globals())

Model exists: False
Test data exists: False


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset_path = "/content/drive/MyDrive/Glaucoma_Project/split"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    dataset_path + "/train",
    classes=["glaucoma", "normal"],
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)

val_data = datagen.flow_from_directory(
    dataset_path + "/val",
    classes=["glaucoma", "normal"],
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_data = datagen.flow_from_directory(
    dataset_path + "/test",
    classes=["glaucoma", "normal"],
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 1455 images belonging to 2 classes.
Found 416 images belonging to 2 classes.
Found 210 images belonging to 2 classes.


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    GlobalAveragePooling2D,
    Dropout,
    Dense
)

# Load pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False

# Create model
model = Sequential([
    Input(shape=(224, 224, 3)),
    Rescaling(scale=2.0, offset=-1.0),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("MobileNetV2 model created successfully!")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MobileNetV2 model created successfully!


In [9]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 18s 384ms/step - accuracy: 0.8811 - loss: 0.2814 - val_accuracy: 0.8654 - val_loss: 0.3025
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 356ms/step - accuracy: 0.8770 - loss: 0.2807 - val_accuracy: 0.8726 - val_loss: 0.2974
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 362ms/step - accuracy: 0.8845 - loss: 0.2672 - val_accuracy: 0.8702 - val_loss: 0.2930
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 365ms/step - accuracy: 0.8832 - loss: 0.2695 - val_accuracy: 0.8702 - val_loss: 0.2905
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 374ms/step - accuracy: 0.8866 - loss: 0.2701 - val_accuracy: 0.8726 - val_loss: 0.2908
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 358ms/step - accuracy: 0.8900 - loss: 0.2672 - val_accuracy: 0.8654 - val_loss: 0.2911
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 369ms/step - accuracy: 0.8942 - loss: 0.2584 - val_accuracy: 0.8726 - val_loss: 0.2905
Epoch 8/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 17s 373ms/step - accuracy: 0.8990 - loss: 0.2473 - val_accu

In [10]:
# Evaluate model using test dataset

test_loss, test_accuracy = model.evaluate(test_data)

print("Test Accuracy:", round(test_accuracy * 100, 2), "%")
print("Test Loss:", round(test_loss, 4))

KeyboardInterrupt: 

In [11]:
# Check model and test data

print("Model available:", "model" in globals())
print("Test data available:", "test_data" in globals())
print("Test images:", test_data.samples if "test_data" in globals() else "Not loaded")

Model available: True
Test data available: True
Test images: 210


In [12]:
test_data.reset()

test_loss, test_accuracy = model.evaluate(
    test_data,
    steps=len(test_data),
    verbose=1
)

print("\n========== MODEL EVALUATION ==========")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 57s 9s/step - accuracy: 0.9095 - loss: 0.2140

========== MODEL EVALUATION ==========
Test Accuracy: 90.95%
Test Loss: 0.2140


In [13]:
import os

save_path = "/content/drive/MyDrive/Glaucoma_Project/mobilenetv2_glaucoma_model.keras"

model.save(save_path)

print("Model saved successfully!")
print("Location:", save_path)
print("File size:", round(os.path.getsize(save_path) / (1024 * 1024), 2), "MB")

Model saved successfully!
Location: /content/drive/MyDrive/Glaucoma_Project/mobilenetv2_glaucoma_model.keras
File size: 9.19 MB


In [14]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Reset test data
test_data.reset()

# Predict test images
predictions = model.predict(test_data, verbose=1)

# Convert scores into class labels
y_pred = (predictions.ravel() >= 0.5).astype(int)

# Actual labels
y_true = test_data.classes

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Glaucoma", "Normal"],
    digits=4,
    zero_division=0
))

print("Accuracy:", round(accuracy_score(y_true, y_pred) * 100, 2), "%")

7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 678ms/step
Confusion Matrix:
[[96  6]
 [13 95]]

Classification Report:
              precision    recall  f1-score   support

    Glaucoma     0.8807    0.9412    0.9100       102
      Normal     0.9406    0.8796    0.9091       108

    accuracy                         0.9095       210
   macro avg     0.9107    0.9104    0.9095       210
weighted avg     0.9115    0.9095    0.9095       210

Accuracy: 90.95 %


In [15]:
import os

model_path = "/content/drive/MyDrive/Glaucoma_Project/mobilenetv2_glaucoma_model.keras"

if os.path.exists(model_path):
    print("Model file exists!")
    print("File size:", round(os.path.getsize(model_path) / (1024 * 1024), 2), "MB")
else:
    print("Model file not found. Saving now...")
    model.save(model_path)
    print("Model saved successfully!")

Model file exists!
File size: 9.19 MB
